Preprocessing i need :
1) tiling 
2) remove data that doesn't contain much information 
3) cloud masking (already done through esri)
4) Band normalisation
5) 



For this part we will be doing tiling we will be following the file with the same id over the years

In [5]:
import os
import shutil

root_path = "C:\\Users\\walaa\\Documents\\urban-evolution-ai\\ml-pipeline\\datasets\\copenhagen"
output_path = os.path.join(root_path, "tiles")
os.makedirs(output_path, exist_ok=True)

# Loop through each folder inside root_path (no digit check)
for year in os.listdir(root_path):
    year_path = os.path.join(root_path, year)
    if not os.path.isdir(year_path):
        continue

    # Skip the "tiles" output folder to avoid recursion
    if year == "tiles":
        continue

    # Loop through quadrant folders
    for quadrant in ["1", "2", "3", "4"]:
        quad_path = os.path.join(year_path, quadrant)
        if not os.path.isdir(quad_path):
            continue

        # Loop through tile images
        for filename in os.listdir(quad_path):
            tile_id, ext = os.path.splitext(filename)
            if ext.lower() not in [".png", ".jpg", ".jpeg"]:
                continue

            # Make tile folder
            tile_folder = os.path.join(output_path, tile_id)
            os.makedirs(tile_folder, exist_ok=True)

            # Add year name to filename
            new_filename = f"{year}{ext}"
            new_filepath = os.path.join(tile_folder, new_filename)

            shutil.copy(os.path.join(quad_path, filename), new_filepath)


removing photoes of sea

In [18]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import numpy as np

def is_sea_tile(img_path, min_blue=0.15, debug=False):
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img, dtype=np.float32) / 255.0

    R = arr[:, :, 0]
    G = arr[:, :, 1]
    B = arr[:, :, 2]

    mean_R = float(R.mean())
    mean_G = float(G.mean())
    mean_B = float(B.mean())

    if debug:
        print(f"[DEBUG] {img_path}")
        print(f"  mean_R = {mean_R:.3f}")
        print(f"  mean_G = {mean_G:.3f}")
        print(f"  mean_B = {mean_B:.3f}")

    # Looser rule:
    #  - blue is dominant
    #  - blue is at least a bit > 0
    is_sea = (mean_B > mean_G) and (mean_B > mean_R * 2) and (mean_B > min_blue)

    if debug:
        print("  is_sea =", is_sea)

    return is_sea


In [ ]:
sea_tile = tiles_root / "out_140284_81990"
candidate_img = None

for fname in os.listdir(sea_tile):
    if fname.lower().endswith((".png", ".jpg", ".jpeg")):
        candidate_img = sea_tile / fname
        break

candidate_img


WindowsPath('C:/Users/walaa/Documents/urban-evolution-ai/ml-pipeline/datasets/copenhagen/tiles/out_140284_81990/ESRI 2020.jpeg')

In [19]:
result = is_sea_tile(candidate_img)
print("RESULT:", result)


RESULT: True


In [20]:
import os
import shutil
from pathlib import Path

tiles_root = Path("C:\\Users\\walaa\\Documents\\urban-evolution-ai\\ml-pipeline\\datasets\\copenhagen\\tiles")

deleted = 0
total = 0

for tile_dir in tiles_root.iterdir():
    if not tile_dir.is_dir():
        continue

    total += 1

    # Pick any image inside this tile folder to test
    candidate_img = None
    for fname in os.listdir(tile_dir):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            candidate_img = tile_dir / fname
            break

    if candidate_img is None:
        # No images? Just skip or delete as you like
        continue

    # Check if this tile is mostly sea
    if is_sea_tile(candidate_img):
        print(f"Deleting sea tile folder: {tile_dir}")
        shutil.rmtree(tile_dir)  # PERMANENT DELETE
        deleted += 1

print(f"Done. Deleted {deleted} / {total} tile folders.")


Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140004_82223
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140004_82224
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140005_82223
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140005_82224
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140006_82223
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140006_82224
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140007_82223
Deleting sea tile folder: C:\Users\walaa\Documents\urban-evolution-ai\ml-pipeline\datasets\copenhagen\tiles\out_140007_82224


we will later check out_140284_81990 to see if it got deleted

In [21]:
import os

folder_path = "../datasets/copenhagen/tiles/tile_out_140284_81990"

if os.path.isdir(folder_path):
    print("Folder exists.")
else:
    print("Folder does NOT exist.")


Folder does NOT exist.


In [3]:
from pathlib import Path
import shutil

tiles_root = Path("./datasets/copenhagen/tiles")

required_files = {
    "ESRI 2014.jpeg",
    "ESRI 2020.jpeg",
    "ESRI 2025.jpeg",
}

deleted = 0
total = 0

for tile_dir in tiles_root.iterdir():
    if not tile_dir.is_dir():
        continue

    total += 1
    files_in_tile = {f.name for f in tile_dir.iterdir() if f.is_file()}
    missing = required_files - files_in_tile

    if missing:
        print(f"Deleting {tile_dir.name} (missing {missing})")
        shutil.rmtree(tile_dir)
        deleted += 1

print(f"\nDeleted {deleted} / {total} tile folders.")


Deleting out_140227_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140228_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140229_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140230_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140231_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140232_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140233_82211 (missing {'ESRI 2014.jpeg'})
Deleting out_140233_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140234_82211 (missing {'ESRI 2014.jpeg'})
Deleting out_140234_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140235_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140236_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140237_82224 (missing {'ESRI 2020.jpeg', 'ESRI 2014.jpeg'})
Deleting out_140238_82212 (missing {'ESRI 2014.jpeg'})
Deleting out_140238_82224 (missi

Band normalisation

In [4]:
import numpy as np
from PIL import Image
import os
from pathlib import Path

def compute_mean_std(tiles_root, filename_pattern):
    """
    Compute mean and std across all tiles for a specific year/source.
    
    Args:
        tiles_root: Path to the tiles folder
        filename_pattern: The filename to look for in each tile folder (e.g., "ESRI 2014.png")
    """
    pixel_sum = np.zeros(3)
    pixel_sq_sum = np.zeros(3)
    n_pixels = 0
    n_images = 0

    tiles_path = Path(tiles_root)
    
    for tile_dir in tiles_path.iterdir():
        if not tile_dir.is_dir():
            continue
        
        # Look for the specific file in this tile folder
        img_path = tile_dir / filename_pattern
        if not img_path.exists():
            continue
            
        try:
            img = np.array(Image.open(img_path).convert("RGB"), dtype=np.float32) / 255.0
            H, W, _ = img.shape
            
            pixel_sum += img.reshape(-1, 3).sum(axis=0)
            pixel_sq_sum += (img.reshape(-1, 3) ** 2).sum(axis=0)
            n_pixels += H * W
            n_images += 1
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

    if n_pixels == 0:
        print(f"Warning: No pixels found for pattern '{filename_pattern}'")
        return None, None
    
    mean = pixel_sum / n_pixels
    std = np.sqrt(pixel_sq_sum / n_pixels - mean**2)
    
    print(f"Processed {n_images} images for '{filename_pattern}'")
    print(f"  Mean RGB: [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
    print(f"  Std RGB:  [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")
    
    return mean, std

# Compute statistics for each dataset
tiles_root = "C:\\Users\\walaa\\Documents\\urban-evolution-ai\\ml-pipeline\\datasets\\copenhagen\\tiles"

print("Computing statistics for ESRI 2014...")
mean_2014, std_2014 = compute_mean_std(tiles_root, "ESRI 2014.jpeg")

print("\nComputing statistics for ESRI 2020...")
mean_2020, std_2020 = compute_mean_std(tiles_root, "ESRI 2020.jpeg")

print("\nComputing statistics for ESRI 2025...")
mean_2025, std_2025 = compute_mean_std(tiles_root, "ESRI 2025.jpeg")

print("\nComputing statistics for OpenStreetMap...")
mean_osm, std_osm = compute_mean_std(tiles_root, "openstreetmap.jpeg")

Computing statistics for ESRI 2014...
Processed 137177 images for 'ESRI 2014.jpeg'
  Mean RGB: [0.3482, 0.3661, 0.2865]
  Std RGB:  [0.1813, 0.1558, 0.1492]

Computing statistics for ESRI 2020...


KeyboardInterrupt: 

In [ ]:
def normalize(img, mean, std):
    img = img.astype(np.float32) / 255.0
    return (img - mean) / std


In [ ]:
# img_2014 = normalize(img_2014, mean_2014, std_2014)
# img_2020 = normalize(img_2020, mean_2020, std_2020)
# img_2025 = normalize(img_2025, mean_2025, std_2025)
# img_osm  = normalize(img_osm,  mean_osm,  std_osm)


semantic segmentation

Create OSM masks

In [12]:
from pathlib import Path
import numpy as np
TILES_ROOT = Path("./datasets/copenhagen/tiles")
MASK_NAME = "osm_mask.png"

deleted = 0

for tile_dir in TILES_ROOT.iterdir():
    if not tile_dir.is_dir():
        continue

    mask_path = tile_dir / MASK_NAME
    if mask_path.exists():
        mask_path.unlink()
        deleted += 1

print(f"Deleted {deleted} OSM mask files.")


Deleted 1079 OSM mask files.


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

TILES_ROOT = Path("./datasets/copenhagen/tiles")
OSM_NAME = "openstreetmap.jpeg"   # <-- adjust if needed
OUT_NAME = "osm_mask.png"

# ---------------------------
# Palette: (class_id, name, RGB)
# ---------------------------
PALETTE = [
    (1,  "Buildings",                 (217, 208, 201)),
    (2,  "Major roads (primary/sec)", (242, 179, 179)),
    (10, "Highways (motorway/trunk)", (232, 146, 162)),
    (11, "Streets (residential)",     (237, 237, 237)),
    (15, "Streets (unclassified)",    (255, 255, 255)),
    (13, "Residential (primary)",     (242, 239, 233)),
    (14, "Residential (alt)",         (224, 223, 223)),
    (16, "Airports",                  (196, 182, 171)),
    (4,  "Woodland/trees",            (174, 209, 160)),
    (5,  "Dense trees",               (173, 209, 158)),
    (6,  "Grass/parks",               (200, 250, 204)),
    (7,  "Bushes/scrub",              (37, 150, 190)),
    (8,  "Meadows",                   (205, 235, 176)),
    (9,  "Shrubbery",                 (200, 215, 171)),
    (12, "Sparse/dry greenery",       (238, 240, 213)),
]

# Priority resolves ambiguity when distances are very close (roads > residential etc.)
# Earlier = higher priority
PRIORITY = [10, 2, 11, 15, 1, 16, 13, 14, 4, 5, 6, 8, 9, 12, 7]

def osm_to_mask_nearest(osm_rgb: np.ndarray, max_dist: float = 50.0, tie_eps: float = 2.0) -> np.ndarray:
    H, W, _ = osm_rgb.shape
    px = osm_rgb.reshape(-1, 3)

    class_ids = np.array([c for c, _, _ in PALETTE], dtype=np.int16)
    protos = np.array([rgb for _, _, rgb in PALETTE], dtype=np.int16)

    # --- SAFE distance computation ---
    diff = px[:, None, :].astype(np.float32) - protos[None, :, :].astype(np.float32)
    dist2 = np.sum(diff ** 2, axis=2)
    dist = np.sqrt(dist2)

    best_idx = np.argmin(dist, axis=1)
    best_dist = dist[np.arange(dist.shape[0]), best_idx]

    mask_flat = class_ids[best_idx].astype(np.uint8)
    mask_flat[best_dist > max_dist] = 0  # background

    # Tie-breaking
    part = np.partition(dist, 1, axis=1)
    second_best_dist = part[:, 1]

    ambiguous = (best_dist <= max_dist) & (np.abs(second_best_dist - best_dist) <= tie_eps)
    if np.any(ambiguous):
        priority_rank = {cid: i for i, cid in enumerate(PRIORITY)}
        for i in np.where(ambiguous)[0]:
            drow = dist[i]
            candidates = np.where(drow <= (best_dist[i] + tie_eps))[0]
            cids = class_ids[candidates]
            mask_flat[i] = min(cids, key=lambda cid: priority_rank.get(int(cid), 10_000))

    return mask_flat.reshape(H, W)


created = 0
skipped = 0

for tile_dir in TILES_ROOT.iterdir():
    if not tile_dir.is_dir():
        continue

    osm_path = tile_dir / OSM_NAME
    out_path = tile_dir / OUT_NAME

    if not osm_path.exists():
        skipped += 1
        continue

    osm = np.array(Image.open(osm_path).convert("RGB"))
    # Tuning tip:
    # - JPEG: try max_dist=45..60
    # - PNG:  try max_dist=20..35
    mask = osm_to_mask_nearest(osm, max_dist=50.0, tie_eps=2.0)

    Image.fromarray(mask).save(out_path)
    created += 1

print(f"Done. Created {created} masks, skipped {skipped} folders (no OSM).")


In [ ]:
TILES_ROOT = Path("./datasets/copenhagen/tiles")
START_FOLDER = "out_140108_82034"

created = 0
skipped = 0

# Get all out_* folders, sorted
out_dirs = sorted(
    d for d in TILES_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("out_")
)

# Skip folders until START_FOLDER
out_dirs = [d for d in out_dirs if d.name >= START_FOLDER]

for group_dir in out_dirs:
    for tile_dir in group_dir.iterdir():
        if not tile_dir.is_dir():
            continue

        osm_path = tile_dir / OSM_NAME
        out_path = tile_dir / OUT_NAME

        if not osm_path.exists():
            skipped += 1
            continue

        osm = np.array(Image.open(osm_path).convert("RGB"))
        mask = osm_to_mask_nearest(osm, max_dist=50.0, tie_eps=2.0)

        Image.fromarray(mask).save(out_path)
        created += 1

print(f"Done. Created {created} masks, skipped {skipped} folders (no OSM).")


debugging why mask is black 

In [11]:
from pathlib import Path
import numpy as np
from PIL import Image

tile_dir = Path("./datasets/copenhagen/tiles").glob("*").__next__()  # picks first folder
mask_path = tile_dir / "osm_mask.png"

m = np.array(Image.open(mask_path))
if m.ndim == 3:
    m = m.squeeze(-1)

print("Tile:", tile_dir.name)
print("Unique values:", np.unique(m))
print("Background %:", (m == 0).mean() * 100)


Tile: out_139881_81883
Unique values: [12]
Background %: 0.0


In [ ]:
from PIL import Image
import numpy as np

# class_id -> RGB (use your palette)
COLOR_MAP = {
    0:(0,0,0),
    1:(217,208,201),
    2:(242,179,179),
    10:(232,146,162),
    11:(237,237,237),
    15:(255,255,255),
    13:(242,239,233),
    14:(224,223,223),
    16:(196,182,171),
    4:(174,209,160),
    5:(173,209,158),
    6:(200,250,204),
    7:(37,150,190),
    8:(205,235,176),
    9:(200,215,171),
    12:(238,240,213),
}

def colorize_mask(mask):
    h, w = mask.shape
    out = np.zeros((h,w,3), dtype=np.uint8)
    for k, rgb in COLOR_MAP.items():
        out[mask == k] = rgb
    return out

mask = np.array(Image.open(mask_path))
if mask.ndim == 3: mask = mask.squeeze(-1)

preview = colorize_mask(mask)
Image.fromarray(preview).save(tile_dir / "osm_mask_preview.png")
print("Saved preview:", tile_dir / "osm_mask_preview.png")

pipeline

In [1]:
import tensorflow as tf
from pathlib import Path

TILES_ROOT = Path("../datasets/copenhagen/tiles")
ESRI_NAME = "ESRI 2025.jpeg"
MASK_NAME = "osm_mask.png"

IMG_SIZE = (256, 256)
NUM_CLASSES = 5  # must match your mask classes (0..4)

def list_pairs(tiles_root: Path):
    esri_paths = []
    mask_paths = []
    for d in tiles_root.iterdir():
        if not d.is_dir():
            continue
        e = d / ESRI_NAME
        m = d / MASK_NAME
        if e.exists() and m.exists():
            esri_paths.append(str(e))
            mask_paths.append(str(m))
    return esri_paths, mask_paths

esri_paths, mask_paths = list_pairs(TILES_ROOT)
print("Pairs:", len(esri_paths))

def load_esri_and_mask(esri_path, mask_path):
    # --- ESRI image ---
    img_bytes = tf.io.read_file(esri_path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, IMG_SIZE, method="bilinear")
    img = tf.cast(img, tf.float32) / 255.0  # (H,W,3)

    # --- Mask (single-channel PNG with class ids) ---
    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask_bytes, channels=1)  # (H,W,1)
    mask = tf.image.resize(mask, IMG_SIZE, method="nearest")
    mask = tf.cast(mask, tf.int32)
    mask = tf.squeeze(mask, axis=-1)  # (H,W)

    return img, mask

ds = tf.data.Dataset.from_tensor_slices((esri_paths, mask_paths))
ds = ds.shuffle(buffer_size=min(len(esri_paths), 2000), reshuffle_each_iteration=True)
ds = ds.map(load_esri_and_mask, num_parallel_calls=tf.data.AUTOTUNE)

# Train/val split
val_frac = 0.15
val_size = int(len(esri_paths) * val_frac)
val_ds = ds.take(val_size).batch(8).prefetch(tf.data.AUTOTUNE)
train_ds = ds.skip(val_size).batch(8).prefetch(tf.data.AUTOTUNE)


FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\datasets\\copenhagen\\tiles'

semantic segmentation using unet

In [ ]:
from tensorflow.keras import layers, Model

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def unet(input_shape=(256, 256, 3), num_classes=5, base=32):
    inputs = layers.Input(shape=input_shape)

    c1 = conv_block(inputs, base)
    p1 = layers.MaxPool2D()(c1)

    c2 = conv_block(p1, base * 2)
    p2 = layers.MaxPool2D()(c2)

    c3 = conv_block(p2, base * 4)
    p3 = layers.MaxPool2D()(c3)

    b  = conv_block(p3, base * 8)

    u3 = layers.UpSampling2D()(b)
    u3 = layers.Concatenate()([u3, c3])
    c4 = conv_block(u3, base * 4)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = conv_block(u2, base * 2)

    u1 = layers.UpSampling2D()(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = conv_block(u1, base)

    # logits (no activation); use SparseCategoricalCrossentropy(from_logits=True)
    outputs = layers.Conv2D(num_classes, 1, padding="same")(c6)

    return Model(inputs, outputs)

model = unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=NUM_CLASSES, base=32)
model.summary()


train 

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)


In [ ]:

rgba(37, 150, 190)